In [ ]:
!nvidia-smi

Fri Apr 24 14:25:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers rank_bm25 faiss-cpu joblib evaluate rouge_score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.4 MB/s eta 0:00:00


In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [ ]:
import evaluate

rouge = evaluate.load('rouge')
chrf = evaluate.load('chrf')

def calculate_metrics(predictions, references):
    rouge_results = rouge.compute(predictions=predictions, references=references)

    chrf_results = chrf.compute(predictions=predictions, references=references)

    return {
        "ROUGE-L": round(rouge_results['rougeL'], 4),
        "chrF++": round(chrf_results['score'], 4)
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import numpy as np

def calculate_recall(retrieved_indices, ground_truth_index, k=5):
    top_k_results = retrieved_indices[:k]

    return 1 if ground_truth_index in top_k_results else 0



In [ ]:
import joblib
import faiss
import json
from google.colab import drive

drive.mount('/content/drive')
path = "/content/drive/MyDrive/Studia_MJ/LLM_projekt"

data = joblib.load(f'{path}/data_pack.joblib')
documents = data['documents']
bm25 = data['bm25']

index = faiss.read_index(f"{path}/faiss_index.bin")

with open(f'{path}/synthetic_test_set.json', 'r', encoding='utf-8') as f:
    test_set = json.load(f)

print("Data is loaded")

Mounted at /content/drive
Data is loaded


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
def generate_answer_variant_a(question, model, tokenizer):
    messages = [
        {"role": "system", "content": "You are a helpful beauty assistant expert in cosmetic ingredients."},
        {"role": "user", "content": question}
    ]

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    return response.strip()

In [ ]:
from tqdm import tqdm

results_variant_a = []


for item in tqdm(test_set[:40]):
    question = item['question']
    reference = item['expected_answer']

    prediction = generate_answer_variant_a(question, model, tokenizer)

    results_variant_a.append({
        "question": question,
        "reference": reference,
        "prediction": prediction
    })


preds = [r['prediction'] for r in results_variant_a]
refs = [r['reference'] for r in results_variant_a]

metrics_a = calculate_metrics(preds, refs)

print("\n--- Variant A (Baseline) ---")
print(f"ROUGE-L: {metrics_a['ROUGE-L']}")
print(f"chrF++:  {metrics_a['chrF++']}")

100%|██████████| 40/40 [13:41<00:00, 20.55s/it]



--- Variant A (Baseline) ---
ROUGE-L: 0.1497
chrF++:  32.894


In [ ]:
def search_bm25(query, bm25, k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_n_indices = np.argsort(scores)[::-1][:k]
    return top_n_indices.tolist()

In [ ]:
def generate_answer_variant_b(question, context, model, tokenizer):
    prompt_with_context = f"""
    Answer the user question based ONLY on the provided context.
    If the information is not in the context, say that you don't know.

    Context:
    {context}

    Question:
    {question}
    """

    messages = [
        {"role": "system", "content": "You are a factual cosmetic assistant. Use the provided context to answer questions."},
        {"role": "user", "content": prompt_with_context}
    ]

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

In [ ]:
test_set[1]

{'question': 'What is the source of Leuconostoc/Radish Root Ferment Filtrate and what makes it a good preservative in cosmetics?',
 'expected_answer': 'Leuconostoc/Radish Root Ferment Filtrate is sourced from radishes fermented with Leuconostoc kimchii, a lactic acid bacteria. The antimicrobial peptide secreted during the fermentation process gives it significant preservative properties, making it a promising natural alternative for use in cosmetics, though its effectiveness is lower than common chemical preservatives like parabens or phenoxyethanol.',
 'document_i': 154}

In [ ]:
results_variant_b = []


for item in tqdm(test_set[:40]):
    query = item['question']
    reference = item['expected_answer']
    gt_index = item['document_i']

    indices_b = search_bm25(query, bm25, k=5)

    retrieved_docs = [documents[i] for i in indices_b]
    context = "\n---\n".join(retrieved_docs)

    prediction = generate_answer_variant_b(query, context, model, tokenizer)
    print("query: ", query, "   prediction: ", prediction)

    recall_val = 1 if gt_index in indices_b else 0

    results_variant_b.append({
        "question": query,
        "reference": reference,
        "prediction": prediction,
        "recall": recall_val
    })

preds_b = [r['prediction'] for r in results_variant_b]
refs_b = [r['reference'] for r in results_variant_b]
recalls_b = [r['recall'] for r in results_variant_b]

metrics_b = calculate_metrics(preds_b, refs_b)
avg_recall_b = np.mean(recalls_b)

print("\n--- Variant B (BM25) ---")
print(f"ROUGE-L:   {metrics_b['ROUGE-L']}")
print(f"chrF++:    {metrics_b['chrF++']}")
print(f"Recall@5:  {avg_recall_b:.4f}")

  2%|▎         | 1/40 [00:04<03:06,  4.78s/it]

query:  What is the primary function of Methyl Gluceth-10 in cosmetic formulations?    prediction:  Methyl Gluceth-10 is a humectant ingredient that helps the skin to cling onto water, giving it a smooth, silky feel. It also reduces the tackiness of other humectants.


  5%|▌         | 2/40 [00:11<03:51,  6.10s/it]

query:  What is the source of Leuconostoc/Radish Root Ferment Filtrate and what makes it a good preservative in cosmetics?    prediction:  The source of Leuconostoc/Radish Root Ferment Filtrate is radishes fermented with Leuconostoc kimchii, a lactic acid bacteria. It is a good preservative in cosmetics due to the significant antimicrobial properties of a peptide secreted from the bacteria during the fermentation process.


  8%|▊         | 3/40 [00:18<04:01,  6.53s/it]

...

 98%|█████████▊| 39/40 [04:43<00:07,  7.38s/it]

query:  What is the primary function of palmitic acid in skincare products, and in what types of products can it be found according to the document content?    prediction:  I don't know.


100%|██████████| 40/40 [04:51<00:00,  7.29s/it]

query:  What are the potential risks associated with Symphytum Officinale Root Extract in cosmetics and how are they regulated in Germany?    prediction:  The potential risk associated with Symphytum Officinale Root Extract in cosmetics is the presence of pyrrolizidine alkaloids (PA), which have proven toxic effects. In Germany, the permissible dose of PA is 100 μg/day, and the treatment is limited to 4-6 weeks per year.



--- Variant B (BM25) ---
ROUGE-L:   0.4759
chrF++:    59.074
Recall@5:  0.8500


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss


embed_model = SentenceTransformer("microsoft/harrier-oss-v1-0.6b")
embeddings = embed_model.encode(documents,
    batch_size=4,
    show_progress_bar=True,
    convert_to_numpy=True
)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"Index FAISS size: {index.ntotal}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/255 [00:00<?, ?it/s]

Index FAISS size: 1020


In [ ]:
def search_faiss(query, embed_model, index, k=5):
  query_vector = embed_model.encode([query]).astype('float32')
  distances, indices = index.search(query_vector, k)
  return indices[0].tolist()

In [ ]:
results_variant_c = []


for item in tqdm(test_set[:40]):
    query = item['question']
    reference = item['expected_answer']
    gt_index = item['document_i']

    retrieved_indices = search_faiss(query, embed_model, index, k=5)

    retrieved_docs = [documents[i] for i in retrieved_indices]
    context = "\n---\n".join(retrieved_docs)

    prediction = generate_answer_variant_b(query, context, model, tokenizer)
    print("query: ", query, "   prediction: ", prediction)

    recall_val = 1 if gt_index in retrieved_indices else 0

    results_variant_b.append({
        "question": query,
        "reference": reference,
        "prediction": prediction,
        "recall": recall_val
    })

preds_c = [r['prediction'] for r in results_variant_b]
refs_c = [r['reference'] for r in results_variant_b]
recalls_c = [r['recall'] for r in results_variant_b]

metrics_c = calculate_metrics(preds_c, refs_c)
avg_recall_c = np.mean(recalls_c)

print("\n--- Variant C (Dense) ---")
print(f"ROUGE-L:   {metrics_c['ROUGE-L']}")
print(f"chrF++:    {metrics_c['chrF++']}")
print(f"Recall@5:  {avg_recall_c:.4f}")

  2%|▎         | 1/40 [00:07<04:35,  7.08s/it]

query:  What is the primary function of Methyl Gluceth-10 in cosmetic formulations?    prediction:  The primary function of Methyl Gluceth-10 is a humectant ingredient that helps the skin to cling onto water.


  5%|▌         | 2/40 [00:27<09:20, 14.76s/it]

query:  What is the source of Leuconostoc/Radish Root Ferment Filtrate and what makes it a good preservative in cosmetics?    prediction:  The source of Leuconostoc/Radish Root Ferment Filtrate is radishes fermented with Leuconostoc kimchii, a lactic acid bacteria. It has significant antimicrobial properties due to a peptide secreted from the bacteria during the fermentation process, making it a good preservative in cosmetics.


  8%|▊         | 3/40 [00:45<10:04, 16.35s/it]

...

 98%|█████████▊| 39/40 [05:36<00:07,  7.99s/it]

query:  What is the primary function of palmitic acid in skincare products, and in what types of products can it be found according to the document content?    prediction:  According to the context, the primary function of palmitic acid in skincare products is that it can make the skin feel nice and smooth in moisturizers (emollient) or it can act as a foam building cleansing agent in cleansers.


100%|██████████| 40/40 [05:43<00:00,  8.59s/it]

query:  What are the potential risks associated with Symphytum Officinale Root Extract in cosmetics and how are they regulated in Germany?    prediction:  The potential risk associated with Symphytum Officinale Root Extract in cosmetics is the presence of pyrrolizidine alkaloids (PA), which have proven toxic effects. In Germany, the permissible dose of PA is 100 μg/day, and the treatment is limited to 4-6 weeks per year.



--- Variant C (Dense) ---
ROUGE-L:   0.5064
chrF++:    61.7682
Recall@5:  0.9500
